In [39]:
from pydantic import BaseModel, Field, NonNegativeInt
from typing import List, Optional

In [40]:
class DocumentPDF(BaseModel):
    name: str
    pages: List[int]
    pageLabels: List[str]
    data: str
    

class Citation(BaseModel):
    documentName: str = Field(
        description='The full name of the pdf document'
    )
    pageLabels: List[str] = Field(
        description=(
            'List of the pdf page labels (the printed page numbers, as opposed to the internal PDF page indices). '
            'Each label must correspond positionally to the matching entry in pageIndices.'
        )
    )
    pageIndices: List[NonNegativeInt] = Field(
        description=(
            'List of 0-based page indices (internal PDF position, NOT the printed page number). '
            'Index 0 = first page of the PDF, index 1 = second page, etc. '
            'Each index must correspond positionally to the matching entry in pageLabels.'
        )
    )


class ResponseOutput(BaseModel):
    response: str = Field(
        description="The answer generated by the model"
    )
    summaryTitle: str = Field(
        description="A brief title summarizing the answer"
    )
    citations: List[Citation] = Field(
        description="List of citations used in the answer"
    )

    def __contains__(self, document_name: str) -> bool:
        for citation in self.citations:
            if citation.documentName == document_name:
                return True
        return False
    
    def __getitem__(self, document_name: str) -> Citation:
        for citation in self.citations:
            if citation.documentName == document_name:
                return citation
        
        raise KeyError(f"Document name '{document_name}' not found in citations.")
    
    def update_citation_pages(self, new_pdf_pages: list[int], new_page_labels: list[str], document_name: str) -> None:
        if document_name not in self:
            return
        
        citation = self[document_name]
        if citation is None:
            return
        
        try:
            indices = citation.pageIndices
            citation.pageIndices = [new_pdf_pages[index] + 1 for index in indices]
            citation.pageLabels = [new_page_labels[index] for index in indices]
        except:
            print('Not able to write citation!')

In [81]:
def format_citation_pages(response: ResponseOutput, documents: list[DocumentPDF]):
    for pdf in documents:
        if not pdf.name in response:
            continue

        citation = response[pdf.name]
        try:
            new_page_labels = [pdf.pageLabels[index] for index in citation.pageIndices]
            new_page_indices = [pdf.pages[index] + 1 for index in citation.pageIndices]
            print("Updating using page indices...")
        except:
            new_page_indices = [index + 1 for index in pdf.pages if pdf.pageLabels[index] in citation.pageLabels]
            new_page_labels = [pdf.pageLabels[index] for index in pdf.pages if pdf.pageLabels[index] in citation.pageLabels]
            print("Updating using page labels...")
        
        if not new_page_indices or not new_page_labels:
            new_page_indices = citation.pageIndices
            new_page_labels = citation.pageLabels
            print(f"Not able to update citation for document '{pdf.name}'!")

        citation.pageIndices = new_page_indices
        citation.pageLabels = new_page_labels

In [86]:
response = ResponseOutput(
    response="This is an answer.",
    summaryTitle="Answer Summary",
    citations=[
        Citation(
            documentName="example.pdf",
            pageLabels=["1", "2"],
            pageIndices=[3,4]
        )
    ]
)

documents = [
    DocumentPDF(
        name="example.pdf",
        pages=[0, 1, 2],
        pageLabels=["1", "2", "3"],
        data="PDF data here"
    )
]

In [87]:
format_citation_pages(response, documents)

print(response)

Updating using page labels...
response='This is an answer.' summaryTitle='Answer Summary' citations=[Citation(documentName='example.pdf', pageLabels=['1', '2'], pageIndices=[1, 2])]
